# Introduction

# Stage 06 — Freshness sensitivity and the player-level audit

**Pipeline position:** sixth. Reads the eligible rows and the row-level frame; appends the Underdog
summary rows and writes the audit block.

## Part 1 — the freshness confound

`fantasy/seasonal_projections/PREREGISTRATION.md` records a named confound under **SLEEPER FRESHNESS
ASYMMETRY**: the stored Sleeper projection is a **week-1-eve snapshot**, while Sleeper ADP is a
late-frozen aggregate of drafts held across the whole summer **with no timestamp**. So part of any
measured edge may be late news — camp injuries, depth-chart decisions — that entered the projection
*after* a share of the ADP sample had already drafted. The prereg states this attaches to **every**
Sleeper-vs-ADP comparison in the repo, and that includes this study.

That makes the primary result a **late-draft, board-analogue signal**, not a clean forecast made when
the market formed.

**The sensitivity.** Replace the untimestamped Sleeper ADP with a *dated* market: the final Underdog
best-ball draft window (W10, or W9 for 2024 where W10 is empty), reconstructed from
`h11_freshness_signal.py`'s own staged, SHA-256-manifested draft dumps. Both sides then reference a
market that existed on a known date.

**Constraints honoured.** `verify_manifest()` re-checks all 16 staged files before anything is read;
the loader touches only local CSVs (**no network**); nothing is written back to any research
artifact; the frozen H11/H12 results are neither re-fired nor altered.

**Why it stays a separate table.** Underdog best ball is a different format from Sleeper half-PPR
redraft — different scoring, different roster construction, therefore different prices. This is a
directional robustness reading, not a replacement result.

## Part 2 — the player-level audit

Aggregates can hide a cell that is arithmetically correct and practically meaningless. This part
prints the individual drafted-board calls — largest buy hits, largest fade hits, and the largest
misses by consensus strength — so the result can be judged by inspection.

**Scope is the drafted board deliberately.** Stage 03 established that the full-population examples
are undrafted players scoring near zero: useless as evidence and dangerous as content.

**A hard boundary.** These are completed 2024 and 2025 seasons auditing a backtest. They are **not**
2026 recommendations, and this study validates no player-level call at any threshold.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | `interim/eligible_rows.csv`, `artifacts/player_season_results.csv`, `h11_freshness_signal.py` + its staged dumps |
| out | `artifacts/threshold_summary.csv` (Underdog rows appended, idempotently), `interim/stage06_underdog.json` |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** `interim/eligible_rows.csv`, `artifacts/player_season_results.csv`, and h11's staged Underdog dumps
**and writes:** the Underdog rows of `artifacts/threshold_summary.csv` and `interim/stage06_underdog.json`

### Explain — reconstruct the dated Underdog market and re-run the thresholds against it

The freshness confound, confronted with data.

`PREREGISTRATION.md` records **SLEEPER FRESHNESS ASYMMETRY**: the stored Sleeper projection is a
week-1-eve snapshot, while Sleeper ADP is a late-frozen summer aggregate with no timestamp. Part of
any measured edge may therefore be late news that entered the projection after some of the ADP sample
had already drafted.

**The sensitivity.** Replace the untimestamped Sleeper ADP with a *dated* market — the final Underdog
best-ball draft window (W10, or W9 for 2024 where W10 is empty) — reconstructed from
`h11_freshness_signal.py`'s own staged, SHA-256-manifested dumps. Both sides of the comparison then
reference a market that existed on a known date. Ranks and signals are rebuilt from scratch on the new
price using the same shared-library functions, so nothing but the market changes.

**Constraints honoured, and checked.** `verify_manifest()` re-checks all 16 staged files before
anything is read; the loader touches only local CSVs, **no network**; nothing is written back to any
research artifact; the frozen H11/H12 results are neither re-fired nor altered.

**Idempotence.** The Underdog rows are appended to `artifacts/threshold_summary.csv`, so the cell
first drops any existing `underdog_final_window` rows. Re-running stage 06 twice cannot duplicate
them.

**Kept separate on purpose.** Underdog best ball has different scoring and roster construction than
Sleeper half-PPR redraft, so this is a directional robustness reading, not a replacement result.

In [2]:
ELIGIBLE = pd.read_csv(INTERIM / "eligible_rows.csv")
PLAYER_RESULTS = pd.read_csv(ARTIFACTS / "player_season_results.csv")
SIGNALS = PLAYER_RESULTS.rename(columns={"position": "pos", "model_pred": "pred",
                                         "actual_half_ppr": "y", "model_family": "model"})
SUMMARY = pd.read_csv(ARTIFACTS / "threshold_summary.csv")
print(f"loaded {len(ELIGIBLE):,} eligible rows, {len(SIGNALS):,} row-level records, "
      f"{len(SUMMARY):,} summary cells\n")

UNDERDOG = {"status": "not_attempted"}
UD_SUMMARY = pd.DataFrame()
try:
    sys.path.insert(0, str(SEAS_DIR))
    import h11_freshness_signal as H11
    H11.verify_manifest()

    _uframes = []
    for yr in H11.PANEL:
        wadp, counts = H11.load_windows(yr)
        win = H11.FINAL_WIN[yr]
        w = wadp[wadp.grp == win].copy()
        w["season"] = yr
        _uframes.append(w)
        print(f"  {yr}: final window {win:>3}  players priced {len(w):>4}  "
              f"drafts in window {counts.get(win, 0):>6,}")
    UD = (pd.concat(_uframes, ignore_index=True).rename(columns={"pos_n": "position"})
          .drop_duplicates(["season", "nn", "position"]))

    ud_elig = ELIGIBLE.merge(UD[["season", "nn", "position", "ud_adp"]],
                             left_on=["season", "norm_name", "pos"],
                             right_on=["season", "nn", "position"], how="left")
    _cov_all = float(ud_elig.ud_adp.notna().mean())
    _d180 = ud_elig[ud_elig.adp_overall_rank <= DRAFTABLE_POOL_SIZE]
    _cov_draft = float(_d180.ud_adp.notna().mean())
    ud_elig = ud_elig[ud_elig.ud_adp.notna()].copy()
    ud_elig["adp_half_ppr"] = ud_elig["ud_adp"]        # the dated market replaces the untimed price

    _rows = []
    for pop_name, cap in POPULATIONS.items():
        base = population_slice(ud_elig, cap)
        for uni in ("A", "B"):
            d = add_signals(build_ranks(base, uni))
            d = d[d.complete]
            for panel in POOLED_PANELS:
                p = d[d.season.isin(PANELS[panel])]
                for t in THRESHOLDS:
                    _rows.append(summarise_cell(p[p[thr_col(t)]], {
                        "market": "underdog_final_window", "population": pop_name, "universe": uni,
                        "panel": panel, "threshold": t, "split": "all", "split_value": "all",
                        "panel_complete_n": len(p)}))
    UD_SUMMARY = pd.DataFrame(_rows)
    UNDERDOG = {"status": "reconstructed",
                "final_window": {str(k): v for k, v in H11.FINAL_WIN.items()},
                "match_rate_all_adp": _cov_all, "match_rate_drafted_top180": _cov_draft,
                "matched_rows": int(len(ud_elig)),
                "matched_rows_drafted_top180": int(_d180.ud_adp.notna().sum()),
                "drafted_top180_rows": int(len(_d180)),
                "manifest_files_verified": 16, "network_used": False}
    print(f"\njoin coverage: all_adp {_cov_all:.1%}  |  drafted_top180 {_cov_draft:.1%} "
          f"({int(_d180.ud_adp.notna().sum())}/{len(_d180)})")

    print("\n" + "=" * 118)
    print("SLEEPER ADP  vs  DATED UNDERDOG FINAL WINDOW — agreement hit rate, universe A")
    print("=" * 118)
    for panel in POOLED_PANELS:
        print(f"\n-- {panel} --")
        rows = []
        for pop_name in POPULATIONS:
            for t in THRESHOLDS:
                s = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.population == pop_name)
                            & (SUMMARY.universe == "A") & (SUMMARY.panel == panel)
                            & (SUMMARY.threshold == t) & (SUMMARY.split == "all")].iloc[0]
                u = UD_SUMMARY[(UD_SUMMARY.population == pop_name) & (UD_SUMMARY.universe == "A")
                               & (UD_SUMMARY.panel == panel) & (UD_SUMMARY.threshold == t)].iloc[0]
                rows.append({"population": pop_name, "threshold": t, "sleeper_n": s.n,
                             "sleeper_hr": s.hit_rate, "underdog_n": u.n, "underdog_hr": u.hit_rate,
                             "delta_hr": u.hit_rate - s.hit_rate, "ud_wilson_lo": u.wilson_lo,
                             "ud_wilson_hi": u.wilson_hi, "ud_too_small": u.too_small_n_lt_10})
        print(pd.DataFrame(rows).round(4).to_string(index=False))

    _deltas = []
    for panel in POOLED_PANELS:
        for pop_name in POPULATIONS:
            for t in THRESHOLDS:
                s = SUMMARY[(SUMMARY.market == "sleeper_adp") & (SUMMARY.population == pop_name)
                            & (SUMMARY.universe == "A") & (SUMMARY.panel == panel)
                            & (SUMMARY.threshold == t) & (SUMMARY.split == "all")].hit_rate.iloc[0]
                u = UD_SUMMARY[(UD_SUMMARY.population == pop_name) & (UD_SUMMARY.universe == "A")
                               & (UD_SUMMARY.panel == panel) & (UD_SUMMARY.threshold == t)].hit_rate.iloc[0]
                _deltas.append(u - s)
    UNDERDOG["comparisons"] = len(_deltas)
    UNDERDOG["all_attenuate"] = bool(all(d < 0 for d in _deltas))
    print(f"\nATTENUATION: {sum(1 for d in _deltas if d < 0)} of {len(_deltas)} comparisons "
          f"attenuate against the dated market (range {min(_deltas):+.4f} to {max(_deltas):+.4f})")

    SUMMARY = pd.concat([SUMMARY[SUMMARY.market != "underdog_final_window"], UD_SUMMARY],
                        ignore_index=True)      # idempotent: drop-then-append
    SUMMARY.to_csv(ARTIFACTS / "threshold_summary.csv", index=False)
    print(f"appended {len(UD_SUMMARY)} Underdog rows -> artifacts/threshold_summary.csv "
          f"({len(SUMMARY):,} rows total)")
except Exception as exc:
    UNDERDOG = {"status": "BLOCKED", "error": f"{type(exc).__name__}: {exc}"}
    print(f"UNDERDOG CHECK BLOCKED: {UNDERDOG['error']}")

(INTERIM / "stage06_underdog.json").write_text(json.dumps(UNDERDOG, indent=2), encoding="utf-8")
print("wrote interim/stage06_underdog.json")

loaded 1,883 eligible rows, 5,315 row-level records, 1,298 summary cells



  assert staged-file sha256s match J1 manifest (16/16): PASS


  2021: final window W10  players priced  392  drafts in window  1,826


  2022: final window W10  players priced  463  drafts in window  5,381


  2023: final window W10  players priced  440  drafts in window  5,553


  2024: final window  W9  players priced  446  drafts in window  7,663


  2025: final window W10  players priced  423  drafts in window  3,031

join coverage: all_adp 77.6%  |  drafted_top180 99.5% (882/886)

SLEEPER ADP  vs  DATED UNDERDOG FINAL WINDOW — agreement hit rate, universe A

-- pooled_2024_2025 --
    population  threshold  sleeper_n  sleeper_hr  underdog_n  underdog_hr  delta_hr  ud_wilson_lo  ud_wilson_hi  ud_too_small
       all_adp        0.0        512      0.7988         451       0.7317   -0.0671        0.6890        0.7705         False
       all_adp        5.0        338      0.8876         245       0.8122   -0.0753        0.7587        0.8562         False
       all_adp        7.5        307      0.9023         204       0.8431   -0.0591        0.7869        0.8866         False
       all_adp       10.0        259      0.9228         163       0.8650   -0.0577        0.8041        0.9092         False
drafted_top180        0.0        162      0.6605         172       0.6279   -0.0326        0.5536        0.6966         False
draft


ATTENUATION: 24 of 24 comparisons attenuate against the dated market (range -0.2308 to -0.0252)
appended 48 Underdog rows -> artifacts/threshold_summary.csv (1,346 rows total)
wrote interim/stage06_underdog.json


### Interpretation — against a dated market the signal attenuates everywhere but survives

The reconstruction ran offline and clean: **16 of 16 staged files matched their SHA-256 manifest**,
and the final windows carry 1,826 to 7,663 drafts each, pricing 392–463 players per season. Join
coverage is **99.5% on the drafted board** (882 of 886) and 77.6% on the full population — the deep
tail is where Underdog and Sleeper disagree about who is worth pricing at all, consistent with
everything stage 03 found about that region.

**Every single comparison attenuates: 24 of 24, no exceptions**, in either population, at any
threshold, in any panel. On the five-season drafted panel: 72.2% → **68.3%** at t>0, 83.8% → **78.3%**
at t>5, 88.6% → **86.1%** at t>7.5, 89.2% → **85.7%** at t>10 (deltas −0.039, −0.055, −0.025, −0.035).
On the full population the deltas run a consistent −0.046 to −0.075.

That uniformity is the informative part. A freshness component appearing in some cells and not others
could be noise; a decline in **all 24** is what a systematic timing advantage looks like. The prereg's
named confound is not hypothetical — the untimestamped Sleeper ADP is measurably easier to beat than
a market with a known date.

**But the signal does not collapse.** At 78.3% and 86.1% on the five-season drafted panel against a
dated market, most of the effect survives. Freshness is a *component*, not the *explanation*.

**How much of a component cannot be measured from this.** The cells are small — the largest drafted
Underdog cell is 129 calls and the 2024–25 t>10 cell is **13 calls**, where the delta reads −0.231 and
means almost nothing. And the format difference is real and uncontrolled: Underdog best ball has
different scoring and roster construction than Sleeper half-PPR redraft, so part of every delta is
format rather than timing. No decomposition is attempted and none should be read in.

**Consequence for the verdict:** the primary result must be labelled a **late-draft, board-analogue
signal**, not clean forecast skill.

### Explain — look at the individual calls

Aggregates can hide a cell that is arithmetically correct and practically meaningless. This cell
prints the individual calls so the result can be judged by inspection rather than taken on trust.

**Scope: the drafted board** (`adp_overall_rank <= 180`), universe A, 2024–2025, agreement at `t>5`.
Deliberately the drafted population — stage 03 established that the full-population examples are
undrafted players scoring near zero, which are useless as evidence and actively dangerous as content.

**Three tables.** Largest buy hits (agreement said "above his price", he finished above it); largest
fade hits (the reverse); and the largest **misses** by absolute consensus strength. The misses matter
more than the hits, because they show the failure modes.

**Columns** carry both the overall ADP and the positional `adp_rank`, all four ranks, both gaps, the
consensus score, the realised `actual_gap`, and the actual half-PPR total — so the size of each call
and the size of the realised move are both visible, and a "hit" that moved two rank spots is
distinguishable from one that moved forty.

**A hard boundary.** These are completed 2024 and 2025 seasons auditing a backtest. They are **not**
2026 recommendations, and this study validates no player-level call at any threshold.

In [3]:
AUDIT_COLS = ["season", "pos", "player", "adp_half_ppr", "adp_rank", "model_rank", "sleeper_rank",
              "actual_rank", "model_gap", "sleeper_gap", "consensus_score", "actual_gap",
              "y", "outcome", "group"]
_RENAME = {"adp_half_ppr": "adp_overall", "y": "actual_pts", "pos": "position"}

_aud = SIGNALS[(SIGNALS.population == "drafted_top180") & (SIGNALS.universe == "A")
               & SIGNALS.complete & SIGNALS.season.isin([2024, 2025]) & SIGNALS[thr_col(5.0)]].copy()

print(f"DRAFTED-BOARD AUDIT — universe A, 2024-2025, agreement at t>5   (n = {len(_aud)} calls)")
print(f"outcome split  : {dict(_aud.outcome.value_counts())}")
print(f"direction split: {dict(_aud.direction.value_counts())}")
print(f"calls per season: {len(_aud)/2:.0f}")

_hits, _miss = _aud[_aud.outcome == "hit"], _aud[_aud.outcome == "miss"]
print("\n" + "=" * 128)
print("LARGEST BUY HITS  (agreement ranked him above his draft price; he finished above it)")
print("=" * 128)
print(_hits.nlargest(12, "consensus_score")[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "=" * 128)
print("LARGEST FADE HITS  (agreement ranked him below his draft price; he finished below it)")
print("=" * 128)
print(_hits.nsmallest(12, "consensus_score")[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "=" * 128)
print("LARGEST MISSES  (by |consensus_score|) — the confident calls that were wrong")
print("=" * 128)
print(_miss.reindex(_miss.consensus_score.abs().sort_values(ascending=False).index)
      .head(15)[AUDIT_COLS].rename(columns=_RENAME).round(1).to_string(index=False))

print("\n" + "-" * 128)
print("How far did the calls actually move, in rank spots?")
print(_aud.groupby("direction").actual_gap.describe()[["count", "mean", "50%", "min", "max"]]
      .round(2).to_string())
print("\nNOTE: 2024/2025 historical outcomes auditing a backtest. NOT 2026 recommendations.")
print("      This study validates no player-level call at any threshold.")

DRAFTED-BOARD AUDIT — universe A, 2024-2025, agreement at t>5   (n = 40 calls)
outcome split  : {'hit': np.int64(34), 'miss': np.int64(5), 'tie': np.int64(1)}
direction split: {'buy': np.int64(22), 'fade': np.int64(18)}
calls per season: 20

LARGEST BUY HITS  (agreement ranked him above his draft price; he finished above it)
 season position            player  adp_overall  adp_rank  model_rank  sleeper_rank  actual_rank  model_gap  sleeper_gap  consensus_score  actual_gap  actual_pts outcome   group
   2024       WR    Michael Wilson        201.7      73.0        55.0          58.0         49.0       18.0         15.0             15.0        24.0       101.0     hit veteran
   2024       RB     Chuba Hubbard        130.6      43.0        29.0          24.0         15.0       14.0         19.0             14.0        28.0       220.1     hit veteran
   2024       WR     Jakobi Meyers        144.5      55.0        29.0          41.0         23.0       26.0         14.0             14.0  

### Interpretation — 40 calls in two seasons, and the misses are the familiar failure modes

The full drafted-board `t>5` cell for 2024–2025 is **40 calls: 34 hits, 5 misses, 1 tie**, split 22
buy / 18 fade — **20 calls per season**. Small enough that a single season's variance moves the hit
rate several points, which is the practical scale this signal operates at on a real draft board.

The **buy hits** are recognisable mid-to-late-round wins: Chuba Hubbard (RB43 by price, finished RB15
with 220.1 points), Bucky Irving (RB59 → RB14, 220.9), Rico Dowdle (RB58 → RB17, 196.8), Jakobi Meyers
(WR55 → WR23), Josh Downs (WR66 → WR34). These are exactly the calls a draft board is *for* — cheap
players both projections liked who returned starter value.

The **fade hits** are equally legible: Marquise Brown (WR41 → WR73, 13.6 points), Zamir White (RB24 →
RB55), Deebo Samuel Sr. (WR14 → WR40), Christian Kirk (WR31 → WR62), Anthony Richardson (QB6 → QB17).
Note a fade hit is not the same as a bust — Deebo still scored 130.1 points; he simply returned less
than his price.

**The five misses are the most useful rows in the study.** Chris Godwin Jr. was a buy who suffered a
season-ending ankle injury in week 7. Garrett Wilson was a buy who finished WR48 from WR18. **Jahmyr
Gibbs was a *fade* of a player priced RB4 who finished RB2 with 336.9 points** — the model's
structural conservatism on elite players, documented in `fantasy/projections/GUIDE.md`, showing up at
the very top of the board. Xavier Worthy and Gabe Davis round it out. So the failure modes are
**availability shocks and elite-tier conservatism** — both already-known weaknesses, and neither
fixable by tightening a threshold.

The move-size table shows the calls that land do move meaningfully: buys gain a mean of **+17.2 rank
spots** (median +19), fades lose **−13.8** (median −12). But the ranges include a buy that fell 30
spots and a fade that gained 3, so the distribution is wide.

# Conclusion and Next Steps

## What this stage established

**Freshness is a real, measured component — and it is not the whole story.**

The reconstruction ran offline and clean: **16 of 16 staged files matched their SHA-256 manifest**,
join coverage **99.5% on the drafted board** (882 of 886). **Every single comparison attenuates —
all 24, no exceptions, in either population, at every threshold, in every panel.** On the five-season
drafted panel: 72.2% → **68.3%** at t>0, 83.8% → **78.3%** at t>5, 88.6% → **86.1%** at t>7.5, 89.2%
→ **85.7%** at t>10.

That uniformity is the informative part. Attenuation in some cells could be noise; a decline in all
24 is what a systematic timing advantage looks like. But at 78.3% and 86.1% against a dated market,
**most of the effect survives** — so freshness is a *component*, not the *explanation*.

**How much cannot be measured here.** The cells are small (the 2024–25 t>10 Underdog cell holds **13
calls**, where the delta reads −0.231 and means almost nothing), and the format difference is real and
uncontrolled — part of every delta is Underdog-vs-Sleeper format, not timing. No decomposition is
attempted and none should be read in.

**The individual calls are legible.** The drafted-board `t>5` cell for 2024–2025 is **40 calls: 34
hits, 5 misses, 1 tie**, split 22 buy / 18 fade — about **20 calls per season**. Buy hits are
recognisable mid-round wins (Chuba Hubbard RB43→RB15, Bucky Irving RB59→RB14, Rico Dowdle RB58→RB17);
fade hits are equally legible (Marquise Brown WR41→WR73, Deebo Samuel WR14→WR40, Anthony Richardson
QB6→QB17).

**The five misses are the most useful rows in the study.** Chris Godwin — a buy, season-ending ankle
injury in week 7. Garrett Wilson — a buy, WR18 → WR48. **Jahmyr Gibbs — a *fade* of a player priced
RB4 who finished RB2 with 336.9 points**, the model's documented elite-tier conservatism appearing at
the top of the board. So the failure modes are **availability shocks and elite-tier conservatism** —
both already-known weaknesses, and neither fixable by tightening a threshold.

## Consequence for the verdict

The primary result must be labelled a **late-draft, board-analogue signal**, not clean forecast skill.

## Next step

Run **`07_synthesis_and_reproducibility.ipynb`** — reconstruct every published number from the
exported rows, re-hash every input, audit all eight notebooks, and state the verdict.